In [17]:
from pathlib import Path
import json
import random
import shutil
import os
import math

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

RAW_DATASET_DIR = Path("sign_dataset")
TRAIN_RAW_DIR = RAW_DATASET_DIR / "train"
VAL_RAW_DIR = RAW_DATASET_DIR / "val"

STREET_RAW_DIR = Path("street_photos")
STREET_PRED_MASKS_DIR = Path("street_pred_masks")
STREET_OVERLAY_DIR = Path("street_overlay")
STREET_GT_MASKS_DIR = Path("street_gt_masks")

YOLO_DATASET_DIR = Path("yolo_sign_dataset")

MODEL_NAME = "yolo11s-seg.pt"
IMG_SIZE = 640
EPOCHS = 30
BATCH = 6
WORKERS = 2
TRAIN_FRACTION = 0.5
MAX_TRAIN_IMAGES = 40000
DEVICE = "mps"
CONF_THRES = 0.25

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

assert TRAIN_RAW_DIR.exists(), f"Не найдена папка {TRAIN_RAW_DIR}"
assert VAL_RAW_DIR.exists(), f"Не найдена папка {VAL_RAW_DIR}"
STREET_RAW_DIR.mkdir(exist_ok=True)
STREET_GT_MASKS_DIR.mkdir(exist_ok=True)

In [18]:
def list_images(folder: Path):
    image_paths = []
    for ext in IMG_EXTS:
        image_paths.extend(sorted(folder.glob(f"*{ext}")))
    return sorted(image_paths)

def find_image_json_pairs(folder: Path):
    pairs = []
    for image_path in list_images(folder):
        candidates = [
            image_path.with_name(image_path.name + "_coco.json"),
            image_path.with_name(image_path.name + ".json"),
            image_path.with_suffix(image_path.suffix + "_coco.json"),
            image_path.with_suffix(image_path.suffix + ".json"),
            image_path.with_suffix(".json"),
            image_path.with_name(image_path.stem + "_coco.json"),
        ]
        json_path = next((p for p in candidates if p.exists()), None)
        if json_path is not None:
            pairs.append((image_path, json_path))
    return pairs

def read_class_ids_only(json_path: Path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return [int(x) for x in data.get("class_ids", [])]

def load_mask_array(data):
    masks = data.get("masks", [])
    arr = np.array(masks)
    if arr.size == 0:
        return None
    if arr.ndim == 2:
        arr = arr[..., None]
    return arr

def extract_instance_masks_and_classes(json_path: Path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    class_ids = [int(x) for x in data.get("class_ids", [])]
    if len(class_ids) == 0:
        return [], []

    arr = load_mask_array(data)
    if arr is None or arr.ndim != 3:
        return [], []

    if arr.shape[-1] == len(class_ids):
        masks = arr
    elif arr.shape[0] == len(class_ids):
        masks = np.transpose(arr, (1, 2, 0))
    else:
        return [], []

    out_masks = []
    out_classes = []
    for i, cls_id in enumerate(class_ids):
        m = (masks[..., i] > 0).astype(np.uint8)
        if m.sum() == 0:
            continue
        out_masks.append(m)
        out_classes.append(cls_id)

    return out_masks, out_classes

def mask_to_yolo_polygons(mask: np.ndarray):
    mask = (mask > 0).astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    h, w = mask.shape
    polygons = []

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 12:
            continue
        epsilon = 0.002 * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True).reshape(-1, 2)
        if len(approx) < 3:
            continue
        poly = []
        for x, y in approx:
            poly.extend([float(x) / w, float(y) / h])
        if len(poly) >= 6:
            polygons.append(poly)

    return polygons

def ensure_link(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    try:
        os.symlink(src.resolve(), dst)
    except Exception:
        shutil.copy2(src, dst)

def collect_class_mapping(train_pairs, val_pairs):
    class_ids = set()
    for _, json_path in tqdm(train_pairs + val_pairs, desc="Сбор class_ids"):
        class_ids.update(read_class_ids_only(json_path))
    class_ids = sorted(class_ids)
    class_to_idx = {cls_id: i for i, cls_id in enumerate(class_ids)}
    names = {i: f"sign_{cls_id}" for cls_id, i in class_to_idx.items()}
    return class_ids, class_to_idx, names

def convert_split_to_yolo(pairs, split_name, out_root: Path, class_to_idx, fraction=1.0, max_images=None):
    pairs = list(pairs)
    if split_name == "train" and fraction < 1.0:
        keep_n = max(1, int(len(pairs) * fraction))
        pairs = random.sample(pairs, keep_n)
    if split_name == "train" and max_images is not None:
        pairs = pairs[:max_images]

    img_out_dir = out_root / "images" / split_name
    lbl_out_dir = out_root / "labels" / split_name
    img_out_dir.mkdir(parents=True, exist_ok=True)
    lbl_out_dir.mkdir(parents=True, exist_ok=True)

    kept = 0
    empty = 0

    for image_path, json_path in tqdm(pairs, desc=f"Конвертация {split_name}"):
        masks, classes = extract_instance_masks_and_classes(json_path)
        label_lines = []

        for mask, cls_id in zip(masks, classes):
            if cls_id not in class_to_idx:
                continue
            polygons = mask_to_yolo_polygons(mask)
            for poly in polygons:
                label_lines.append(f"{class_to_idx[cls_id]} " + " ".join(f"{v:.6f}" for v in poly))

        out_image = img_out_dir / image_path.name
        out_label = lbl_out_dir / f"{image_path.stem}.txt"

        ensure_link(image_path, out_image)
        with open(out_label, "w", encoding="utf-8") as f:
            f.write("\n".join(label_lines))

        kept += 1
        if len(label_lines) == 0:
            empty += 1

    return {
        "split": split_name,
        "images": kept,
        "empty_labels": empty,
        "non_empty_labels": kept - empty
    }

def build_data_yaml(out_root: Path, names: dict):
    import yaml
    yaml_path = out_root / "data.yaml"
    content = {
        "path": str(out_root.resolve()),
        "train": "images/train",
        "val": "images/val",
        "names": names
    }
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(content, f, allow_unicode=True, sort_keys=False)
    return yaml_path

In [19]:
train_pairs = find_image_json_pairs(TRAIN_RAW_DIR)
val_pairs = find_image_json_pairs(VAL_RAW_DIR)

class_ids, class_to_idx, names = collect_class_mapping(train_pairs, val_pairs)

print("Найденные исходные class_ids:", class_ids)
assert len(class_ids) == 8, f"Ожидалось 8 классов, найдено {len(class_ids)}"

if YOLO_DATASET_DIR.exists():
    shutil.rmtree(YOLO_DATASET_DIR)

train_info = convert_split_to_yolo(
    train_pairs,
    "train",
    YOLO_DATASET_DIR,
    class_to_idx,
    fraction=TRAIN_FRACTION,
    max_images=MAX_TRAIN_IMAGES
)

val_info = convert_split_to_yolo(
    val_pairs,
    "val",
    YOLO_DATASET_DIR,
    class_to_idx,
    fraction=1.0,
    max_images=None
)

DATA_YAML = build_data_yaml(YOLO_DATASET_DIR, names)

display(pd.DataFrame([train_info, val_info]))
display(pd.DataFrame({"yolo_class_idx": list(names.keys()), "class_name": list(names.values())}))

Сбор class_ids: 100%|██████████| 2181/2181 [00:01<00:00, 1198.22it/s]


Найденные исходные class_ids: [1, 2, 3, 4, 6, 8, 10, 13]


Конвертация val: 100%|██████████| 127/127 [00:00<00:00, 615.38it/s]


,split,images,empty_labels,non_empty_labels
0,train,1027,37,990
1,val,127,12,115


,yolo_class_idx,class_name
0,0,sign_1
1,1,sign_2
2,2,sign_3
3,3,sign_4
4,4,sign_6
5,5,sign_8
6,6,sign_10
7,7,sign_13


In [20]:
model = YOLO(MODEL_NAME)

train_results = model.train(
    data=str(DATA_YAML),
    imgsz=IMG_SIZE,
    epochs=EPOCHS,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    optimizer="AdamW",
    lr0=0.002,
    weight_decay=0.0005,
    patience=6,
    save=True,
    project="runs",
    name="russian_signs_seg",
    pretrained=True,
    verbose=True
)

Ultralytics 8.4.33 🚀 Python-3.14.2 torch-2.11.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_sign_dataset/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=russian_signs_seg3, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=6, perspective=0.0, p

In [21]:
best_model_path = Path(train_results.save_dir) / "weights" / "best.pt"
best_model = YOLO(str(best_model_path))

val_builtin = best_model.val(
    data=str(DATA_YAML),
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    split="val"
)

print(best_model_path)
print(val_builtin.results_dict)

Ultralytics 8.4.33 🚀 Python-3.14.2 torch-2.11.0 MPS (Apple M4)
YOLO11s-seg summary (fused): 114 layers, 10,069,912 parameters, 0 gradients, 32.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 218.7±123.6 MB/s, size: 96.2 KB)
val: Scanning /Users/squirtle/Desktop/CV/itmo-cv-lab-6/yolo_sign_dataset/labels/val.cache... 127 images, 12 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 127/127 66.6Mit/s 0.0s
val: /Users/squirtle/Desktop/CV/itmo-cv-lab-6/yolo_sign_dataset/images/val/201.jpg: 1 duplicate labels removed
val: /Users/squirtle/Desktop/CV/itmo-cv-lab-6/yolo_sign_dataset/images/val/202.jpg: 3 duplicate labels removed
val: /Users/squirtle/Desktop/CV/itmo-cv-lab-6/yolo_sign_dataset/images/val/203.jpg: 1 duplicate labels removed
val: /Users/squirtle/Desktop/CV/itmo-cv-lab-6/yolo_sign_dataset/images/val/205.jpg: 3 duplicate labels removed
val: /Users/squirtle/Desktop/CV/itmo-cv-lab-6/yolo_sign_dataset/images/val/208.jpg: 5 duplicate labels removed
val: /Users/squirtle/Desktop/CV/itmo-

In [22]:
def yolo_txt_to_union_mask(label_path: Path, image_shape):
    h, w = image_shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)

    if not label_path.exists():
        return mask

    with open(label_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]

    for line in lines:
        parts = line.split()
        if len(parts) < 7:
            continue
        coords = list(map(float, parts[1:]))
        pts = np.array(coords, dtype=np.float32).reshape(-1, 2)
        pts[:, 0] *= w
        pts[:, 1] *= h
        pts = np.round(pts).astype(np.int32)
        cv2.fillPoly(mask, [pts], 1)

    return mask

def predict_union_mask(model, image_path: Path, conf=0.25):
    image = cv2.imread(str(image_path))
    h, w = image.shape[:2]
    pred_mask = np.zeros((h, w), dtype=np.uint8)

    result = model.predict(
        source=str(image_path),
        imgsz=IMG_SIZE,
        conf=conf,
        device=DEVICE,
        verbose=False
    )[0]

    if result.masks is None:
        return pred_mask

    for poly in result.masks.xy:
        pts = np.round(np.array(poly)).astype(np.int32)
        if len(pts) >= 3:
            cv2.fillPoly(pred_mask, [pts], 1)

    return pred_mask

def calc_metrics(gt_mask, pred_mask):
    gt = (gt_mask > 0).astype(np.uint8)
    pred = (pred_mask > 0).astype(np.uint8)

    tp = int(((gt == 1) & (pred == 1)).sum())
    fp = int(((gt == 0) & (pred == 1)).sum())
    fn = int(((gt == 1) & (pred == 0)).sum())

    iou = tp / (tp + fp + fn + 1e-9)
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    l2 = float(np.sqrt(np.mean((gt.astype(np.float32) - pred.astype(np.float32)) ** 2)))

    return {
        "IoU": iou,
        "Precision": precision,
        "Recall": recall,
        "L2": l2
    }

def evaluate_yolo_split(model, images_dir: Path, labels_dir: Path, conf=0.25):
    rows = []
    image_paths = list_images(images_dir)

    for image_path in tqdm(image_paths, desc=f"Оценка {images_dir.name}"):
        image = cv2.imread(str(image_path))
        label_path = labels_dir / f"{image_path.stem}.txt"
        gt_mask = yolo_txt_to_union_mask(label_path, image.shape)
        pred_mask = predict_union_mask(model, image_path, conf=conf)
        m = calc_metrics(gt_mask, pred_mask)
        rows.append({
            "image": image_path.name,
            **m
        })

    df = pd.DataFrame(rows)
    summary = pd.DataFrame([{
        "images": len(df),
        "IoU_mean": df["IoU"].mean(),
        "Precision_mean": df["Precision"].mean(),
        "Recall_mean": df["Recall"].mean(),
        "L2_mean": df["L2"].mean(),
        "IoU>=0.5_%": (df["IoU"] >= 0.5).mean() * 100,
        "IoU>=0.75_%": (df["IoU"] >= 0.75).mean() * 100,
        "IoU>=0.9_%": (df["IoU"] >= 0.9).mean() * 100
    }])
    return df, summary

In [39]:
val_df, val_summary = evaluate_yolo_split(
    best_model,
    YOLO_DATASET_DIR / "images" / "val",
    YOLO_DATASET_DIR / "labels" / "val",
    conf=CONF_THRES
)

display(val_summary)

Оценка val: 100%|██████████| 127/127 [00:07<00:00, 17.04it/s]


,images,IoU_mean,Precision_mean,Recall_mean,L2_mean,IoU>=0.5_%,IoU>=0.75_%,IoU>=0.9_%
0,127,0.639433,0.883753,0.651414,0.501649,81.102362,44.094488,12.598425


In [35]:
def save_street_predictions(model, image_dir: Path, pred_masks_dir: Path, overlay_dir: Path, conf=0.25):
    if pred_masks_dir.exists():
        shutil.rmtree(pred_masks_dir)
    if overlay_dir.exists():
        shutil.rmtree(overlay_dir)

    pred_masks_dir.mkdir(parents=True, exist_ok=True)
    overlay_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for image_path in tqdm(list_images(image_dir), desc="Предразметка street-фото"):
        image = cv2.imread(str(image_path))
        h, w = image.shape[:2]

        result = model.predict(
            source=str(image_path),
            imgsz=IMG_SIZE,
            conf=conf,
            device=DEVICE,
            verbose=False
        )[0]

        pred_mask = np.zeros((h, w), dtype=np.uint8)
        cls_names = []
        confs = []

        if result.masks is not None:
            for poly in result.masks.xy:
                pts = np.round(np.array(poly)).astype(np.int32)
                if len(pts) >= 3:
                    cv2.fillPoly(pred_mask, [pts], 255)

        if result.boxes is not None and len(result.boxes) > 0:
            cls_ids = result.boxes.cls.detach().cpu().numpy().astype(int).tolist()
            confs = result.boxes.conf.detach().cpu().numpy().tolist()
            cls_names = [result.names[c] for c in cls_ids]

        overlay = result.plot()
        cv2.imwrite(str(pred_masks_dir / f"{image_path.stem}.png"), pred_mask)
        cv2.imwrite(str(overlay_dir / f"{image_path.stem}_overlay.jpg"), overlay)

        rows.append({
            "image": image_path.name,
            "detected": len(cls_names),
            "class_names": ", ".join(cls_names),
            "confidences": ", ".join(f"{x:.3f}" for x in confs)
        })

    return pd.DataFrame(rows)

street_pred_df = save_street_predictions(
    best_model,
    STREET_RAW_DIR,
    STREET_PRED_MASKS_DIR,
    STREET_OVERLAY_DIR,
    conf=CONF_THRES
)

display(street_pred_df)

Предразметка street-фото: 100%|██████████| 10/10 [00:12<00:00,  1.21s/it]


,image,detected,class_names,confidences
0,1.jpg,1,sign_1,0.397
1,10.png,1,sign_3,0.628
2,2.png,1,sign_3,0.671
3,3.jpg,2,"sign_3, sign_1","0.633, 0.281"
4,4.jpg,1,sign_1,0.277
5,5.png,1,sign_1,0.319
6,6.png,1,sign_3,0.651
7,7.png,2,"sign_3, sign_1","0.500, 0.424"
8,8.png,1,sign_1,0.340
9,9.png,2,"sign_3, sign_1","0.573, 0.279"
